# Tokenizer

Tokenizer is an independent module that conducts essential data preprocessing for efficiency of LLM

Actually we already covered its simplest version before:

```python
# Tokenizer! (Extremely Simple Char-level Tokenizer)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [ stoi[c] for c in s] # string to integers
decode = lambda l: ''.join(itos[i] for i in l) # integers to string

print(encode("I need to eat something"))
print(decode(encode("I need to eat something")))
```

But char-wise tokenizer above is rather inefficient in terms of training, so we gonna improve it in chunkwise manner.

In [2]:
"아 배고파 죽겠다.".encode("utf-8")

b'\xec\x95\x84 \xeb\xb0\xb0\xea\xb3\xa0\xed\x8c\x8c \xec\xa3\xbd\xea\xb2\xa0\xeb\x8b\xa4.'

In [3]:
# Let's get into Byte-pair algorithm!
# --> Make a frequent pair into a single token

# First thing first, let's implement a bijective tokenizer.
text = """Бесценная моя Варвара Алексеевна!
Вчера я был счастлив, чрезмерно счастлив, донельзя счастлив! Вы хоть раз в жизни, упрямица, меня послушались. Вечером, часов в восемь, просыпаюсь (вы знаете, маточка, что я часочек-другой люблю поспать после должности), свечку достал, приготовляю бумаги, чиню перо, вдруг, невзначай, подымаю глаза, — право, у меня сердце вот так и запрыгало! Так вы-таки поняли, чего мне хотелось, чего сердчишку моему хотелось! Вижу, уголочек занавески у окна вашего загнут и прицеплен к горшку с бальзамином, точнехонько так, как я вам тогда намекал; тут же показалось мне, что и личико ваше мелькнуло у окна, что и вы ко мне из комнатки вашей смотрели, что и вы обо мне думали. И как же мне досадно было, голубчик мой, что миловидного личика-то вашего я не мог разглядеть хорошенько! Было время, когда и мы светло видели, маточка. Не радость старость, родная моя! Вот и теперь всё как-то рябит в глазах; чуть поработаешь вечером, попишешь что-нибудь, наутро и глаза раскраснеются, и слезы текут так, что даже совестно перед чужими бывает. Однако же в воображении моем так и засветлела ваша улыбочка, ангельчик, ваша добренькая, приветливая улыбочка; и на сердце моем было точно такое ощущение, как тогда, как я поцеловал вас, Варенька, — помните ли, ангельчик? Знаете ли, голубчик мой, мне даже показалось, что вы там мне пальчиком погрозили? Так ли, шалунья? Непременно вы это всё опишите подробнее в вашем письме.
Ну, а какова наша придумочка насчет занавески вашей, Варенька? Премило, не правда ли? Сижу ли за работой, ложусь ли спать, просыпаюсь ли, уж знаю, что и вы там обо мне думаете, меня помните, да и сами-то здоровы и веселы. Опустите занавеску — значит, прощайте, Макар Алексеевич, спать пора! Подымете — значит, с добрым утром, Макар Алексеевич, каково-то вы спали, или: каково-то вы в вашем здоровье, Макар Алексеевич? Что же до меня касается, то я, слава Творцу, здорова и благополучна! Видите ли, душечка моя, как это ловко придумано; и писем не нужно! Хитро, не правда ли? А ведь придумочка-то моя! А что, каков я на эти дела, Варвара Алексеевна?"""
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

print("---")
print(text)
print(f"length: {len(text)}")
print("---")
print(tokens)
print(f"length: {len(tokens)}")

---
Бесценная моя Варвара Алексеевна!
Вчера я был счастлив, чрезмерно счастлив, донельзя счастлив! Вы хоть раз в жизни, упрямица, меня послушались. Вечером, часов в восемь, просыпаюсь (вы знаете, маточка, что я часочек-другой люблю поспать после должности), свечку достал, приготовляю бумаги, чиню перо, вдруг, невзначай, подымаю глаза, — право, у меня сердце вот так и запрыгало! Так вы-таки поняли, чего мне хотелось, чего сердчишку моему хотелось! Вижу, уголочек занавески у окна вашего загнут и прицеплен к горшку с бальзамином, точнехонько так, как я вам тогда намекал; тут же показалось мне, что и личико ваше мелькнуло у окна, что и вы ко мне из комнатки вашей смотрели, что и вы обо мне думали. И как же мне досадно было, голубчик мой, что миловидного личика-то вашего я не мог разглядеть хорошенько! Было время, когда и мы светло видели, маточка. Не радость старость, родная моя! Вот и теперь всё как-то рябит в глазах; чуть поработаешь вечером, попишешь что-нибудь, наутро и глаза раскрасне

In [4]:
# Let's get into the By-pair phase:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]): # Pythonic way to iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
# print(stats)
print(sorted(((v,k) for k, v in stats.items()), reverse=True))

[(243, (32, 208)), (169, (208, 176)), (167, (208, 190)), (142, (208, 181)), (98, (208, 184)), (89, (209, 130)), (83, (32, 209)), (79, (208, 186)), (79, (176, 208)), (78, (208, 187)), (77, (208, 189)), (75, (208, 178)), (70, (44, 32)), (69, (181, 208)), (67, (209, 129)), (65, (209, 128)), (65, (208, 188)), (64, (130, 208)), (63, (190, 208)), (61, (189, 208)), (56, (187, 208)), (53, (128, 208)), (52, (209, 135)), (50, (190, 209)), (50, (178, 208)), (50, (176, 209)), (49, (186, 208)), (47, (208, 191)), (47, (188, 208)), (44, (209, 131)), (44, (208, 180)), (41, (190, 32)), (36, (209, 140)), (36, (135, 208)), (35, (181, 209)), (34, (129, 208)), (33, (184, 208)), (31, (191, 208)), (30, (181, 32)), (30, (129, 209)), (29, (209, 143)), (29, (208, 183)), (27, (209, 139)), (27, (208, 179)), (27, (180, 208)), (25, (184, 32)), (25, (183, 208)), (24, (179, 208)), (23, (176, 32)), (22, (208, 177)), (22, (184, 209)), (20, (209, 136)), (20, (131, 208)), (18, (187, 209)), (18, (136, 208)), (16, (191, 20

In [5]:
# We're gonna replace a high-ranking pair into a single token
top_pair = max(stats, key=stats.get) # Ignore the error
top_pair

(32, 208)

In [6]:
def merge(ids, pair, idx):
    # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
    newids = []
    i = 0
    # print(f"length of ids: {len(ids)}")
    while i < len(ids):
        # if we are not at the very last position AND the pair matches, replace it
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            # print(f"hit!: {ids[i]}, {ids[i+1]}")
            newids.append(idx)
            i += 2
        else:
            # print(ids[i])
            newids.append(ids[i])
            i += 1
    return newids
    
# print("1")
# print(merge([5, 6, 6, 7, 9, 1], (6, 6), 99))

# Actual use case
tokens2 = merge(tokens, top_pair, 256)

print(tokens)
print(f"length: {len(tokens)}")
print("---")
print(tokens2)
print(f"length: {len(tokens2)}")


[208, 145, 208, 181, 209, 129, 209, 134, 208, 181, 208, 189, 208, 189, 208, 176, 209, 143, 32, 208, 188, 208, 190, 209, 143, 32, 208, 146, 208, 176, 209, 128, 208, 178, 208, 176, 209, 128, 208, 176, 32, 208, 144, 208, 187, 208, 181, 208, 186, 209, 129, 208, 181, 208, 181, 208, 178, 208, 189, 208, 176, 33, 10, 208, 146, 209, 135, 208, 181, 209, 128, 208, 176, 32, 209, 143, 32, 208, 177, 209, 139, 208, 187, 32, 209, 129, 209, 135, 208, 176, 209, 129, 209, 130, 208, 187, 208, 184, 208, 178, 44, 32, 209, 135, 209, 128, 208, 181, 208, 183, 208, 188, 208, 181, 209, 128, 208, 189, 208, 190, 32, 209, 129, 209, 135, 208, 176, 209, 129, 209, 130, 208, 187, 208, 184, 208, 178, 44, 32, 208, 180, 208, 190, 208, 189, 208, 181, 208, 187, 209, 140, 208, 183, 209, 143, 32, 209, 129, 209, 135, 208, 176, 209, 129, 209, 130, 208, 187, 208, 184, 208, 178, 33, 32, 208, 146, 209, 139, 32, 209, 133, 208, 190, 209, 130, 209, 140, 32, 209, 128, 208, 176, 208, 183, 32, 208, 178, 32, 208, 182, 208, 184, 208, 183,

In [7]:
# Let's iterate the high-rank pairs
# We need to fix the sweetspot to determine / define what 'high-rank' is then.

# Anyway make the text longer to have more representative token statistics

text = """Бесценная моя Варвара Алексеевна!
Вчера я был счастлив, чрезмерно счастлив, донельзя счастлив! Вы хоть раз в жизни, упрямица, меня послушались. Вечером, часов в восемь, просыпаюсь (вы знаете, маточка, что я часочек-другой люблю поспать после должности), свечку достал, приготовляю бумаги, чиню перо, вдруг, невзначай, подымаю глаза, — право, у меня сердце вот так и запрыгало! Так вы-таки поняли, чего мне хотелось, чего сердчишку моему хотелось! Вижу, уголочек занавески у окна вашего загнут и прицеплен к горшку с бальзамином, точнехонько так, как я вам тогда намекал; тут же показалось мне, что и личико ваше мелькнуло у окна, что и вы ко мне из комнатки вашей смотрели, что и вы обо мне думали. И как же мне досадно было, голубчик мой, что миловидного личика-то вашего я не мог разглядеть хорошенько! Было время, когда и мы светло видели, маточка. Не радость старость, родная моя! Вот и теперь всё как-то рябит в глазах; чуть поработаешь вечером, попишешь что-нибудь, наутро и глаза раскраснеются, и слезы текут так, что даже совестно перед чужими бывает. Однако же в воображении моем так и засветлела ваша улыбочка, ангельчик, ваша добренькая, приветливая улыбочка; и на сердце моем было точно такое ощущение, как тогда, как я поцеловал вас, Варенька, — помните ли, ангельчик? Знаете ли, голубчик мой, мне даже показалось, что вы там мне пальчиком погрозили? Так ли, шалунья? Непременно вы это всё опишите подробнее в вашем письме.
Ну, а какова наша придумочка насчет занавески вашей, Варенька? Премило, не правда ли? Сижу ли за работой, ложусь ли спать, просыпаюсь ли, уж знаю, что и вы там обо мне думаете, меня помните, да и сами-то здоровы и веселы. Опустите занавеску — значит, прощайте, Макар Алексеевич, спать пора! Подымете — значит, с добрым утром, Макар Алексеевич, каково-то вы спали, или: каково-то вы в вашем здоровье, Макар Алексеевич? Что же до меня касается, то я, слава Творцу, здорова и благополучна! Видите ли, душечка моя, как это ловко придумано; и писем не нужно! Хитро, не правда ли? А ведь придумочка-то моя! А что, каков я на эти дела, Варвара Алексеевна?
Доложу я вам, маточка моя, Варвара Алексеевна, что спал я сию ночь добрым порядком, вопреки ожиданий, чем и весьма доволен; хотя на новых квартирах, с новоселья, и всегда как-то не спится; всё что-то так, да не так! Встал я сегодня таким ясным соколом — любо-весело! Что это какое утро сегодня хорошее, маточка! У нас растворили окошко; солнышко светит, птички чирикают, воздух дышит весенними ароматами, и вся природа оживляется — ну, и остальное там всё было тоже соответственное; всё в порядке, по-весеннему. Я даже и помечтал сегодня довольно приятно, и всё об вас были мечтания мои, Варенька. Сравнил я вас с птичкой небесной, на утеху людям и для украшения природы созданной. Тут же подумал я, Варенька, что и мы, люди, живущие в заботе и треволнении, должны тоже завидовать беззаботному и невинному счастию небесных птиц, — ну, и остальное всё такое же, сему же подобное; то есть я всё такие сравнения отдаленные делал. У меня там книжка есть одна, Варенька, так в ней то же самое, всё такое же весьма подробно описано. Я к тому пишу, что ведь разные бывают мечтания, маточка. А вот теперь весна, так и мысли всё такие приятные, острые, затейливые, и мечтания приходят нежные; всё в розовом цвете. Я к тому и написал это всё; а впрочем, я это всё взял из книжки. Там сочинитель обнаруживает такое же желание в стишках и пишет —
Зачем я не птица, не хищная птица!

Ну и т. д. Там и еще есть разные мысли, да Бог с ними! А вот куда это вы утром ходили сегодня, Варвара Алексеевна? Я еще и в должность не сбирался, а вы, уж подлинно как пташка весенняя, порхнули из комнаты и по двору прошли такая веселенькая. Как мне-то было весело, на вас глядя! Ах, Варенька, Варенька! вы не грустите; слезами горю помочь нельзя; это я знаю, маточка моя, это я на опыте знаю. Теперь же вам так покойно, да и здоровьем вы немного поправились. Ну, что ваша Федора? Ах, какая же она добрая женщина! Вы мне, Варенька, напишите, как вы с нею там живете теперь и всем ли вы довольны? Федора-то немного ворчлива; да вы на это не смотрите, Варенька. Бог с нею! Она такая добрая.
Я уже вам писал о здешней Терезе, — тоже и добрая и верная женщина. А уж как я беспокоился об наших письмах! Как они передаваться-то будут? А вот как тут послал господь на наше счастие Терезу. Она женщина добрая, кроткая, бессловесная. Но наша хозяйка просто безжалостная. Затирает ее в работу, словно ветошку какую-нибудь.
Ну, в какую же я трущобу попал, Варвара Алексеевна! Ну, уж квартира! Прежде ведь я жил таким глухарем, сами знаете: смирно, тихо; у меня, бывало, муха летит, так и муху слышно. А здесь шум, крик, гвалт! Да ведь вы еще и не знаете, как это всё здесь устроено. Вообразите, примерно, длинный коридор, совершенно темный и нечистый. По правую его руку будет глухая стена, а по левую всё двери да двери, точно нумера, всё так в ряд простираются. Ну, вот и нанимают эти нумера, а в них по одной комнатке в каждом; живут в одной и по двое, и по трое. Порядку не спрашивайте — Ноев ковчег! Впрочем, кажется, люди хорошие, всё такие образованные, ученые. Чиновник один есть (он где-то по литературной части), человек начитанный: и о Гомере, и о Брамбеусе[3], и о разных у них там сочинителях говорит, обо всем говорит, — умный человек! Два офицера живут и всё в карты играют. Мичман живет; англичанин-учитель живет. Постойте, я вас потешу, маточка; опишу их в будущем письме сатирически, то есть как они там сами по себе, со всею подробностию. Хозяйка наша, — очень маленькая и нечистая старушонка, — целый день в туфлях да в шлафроке[4] ходит и целый день всё кричит на Терезу. Я живу в кухне, или гораздо правильнее будет сказать вот как: тут подле кухни есть одна комната (а у нас, нужно вам заметить, кухня чистая, светлая, очень хорошая), комнатка небольшая, уголок такой скромный… то есть, или еще лучше сказать, кухня большая в три окна, так у меня вдоль поперечной стены перегородка, так что и выходит как бы еще комната, нумер сверхштатный; всё просторное, удобное, и окно есть, и всё, — одним словом, всё удобное. Ну, вот это мой уголочек. Ну, так вы и не думайте, маточка, чтобы тут что-нибудь такое иное и таинственный смысл какой был; что вот, дескать, кухня! — то есть я, пожалуй, и в самой этой комнате за перегородкой живу, но это ничего; я себе ото всех особняком, помаленьку живу, втихомолочку живу. Поставил я у себя кровать, стол, комод, стульев парочку, образ повесил. Правда, есть квартиры и лучше, — может быть, есть и гораздо лучшие, — да удобство-то главное; ведь это я всё для удобства, и вы не думайте, что для другого чего-нибудь. Ваше окошко напротив, через двор; и двор-то узенький, вас мимоходом увидишь — всё веселее мне, горемычному, да и дешевле. У нас здесь самая последняя комната, со столом, тридцать пять рублей ассигнациями[5] стоит. Не по карману! А моя квартира стоит мне семь рублей ассигнациями, да стол пять целковых: вот двадцать четыре с полтиною, а прежде ровно тридцать платил, зато во многом себе отказывал; чай пивал не всегда, а теперь вот и на чай и на сахар выгадал. Оно, знаете ли, родная моя, чаю не пить как-то стыдно; здесь всё народ достаточный, так и стыдно. Ради чужих и пьешь его, Варенька, для вида, для тона; а по мне всё равно, я не прихотлив. Положите так, для карманных денег — всё сколько-нибудь требуется — ну, сапожишки какие-нибудь, платьишко — много ль останется? Вот и всё мое жалованье. Я-то не ропщу и доволен. Оно достаточно. Вот уже несколько лет достаточно; награждения тоже бывают. Ну, прощайте, мой ангельчик. Я там купил парочку горшков с бальзаминчиком и гераньку — недорого. А вы, может быть, и резеду любите? Так и резеда есть, вы напишите; да, знаете ли, всё как можно подробнее напишите. Вы, впрочем, не думайте чего-нибудь и не сомневайтесь, маточка, обо мне, что я такую комнату нанял. Нет, это удобство заставило, и одно удобство соблазнило меня. Я ведь, маточка, деньги коплю, откладываю; у меня денежка водится. Вы не смотрите на то, что я такой тихонький, что, кажется, муха меня крылом перешибет. Нет, маточка, я про себя не промах, и характера совершенно такого, как прилично твердой и безмятежной души человеку. Прощайте, мой ангельчик! Расписался я вам чуть не на двух листах, а на службу давно пора. Целую ваши пальчики, маточка, и пребываю
Вашим нижайшим слугою
и вернейшим другом
Макаром Девушкиным.
P. S. Об одном прошу: отвечайте мне, ангельчик мой, как можно подробнее. Я вам при сем посылаю, Варенька, фунтик конфет; так вы их скушайте на здоровье, да, ради Бога, обо мне не заботьтесь и не будьте в претензии. Ну, так прощайте же, маточка.
"""

tokens = text.encode("utf-8")
tokens = list(map(int, tokens))
print(f"length: {len(tokens)}")

length: 15344


In [8]:
import math

# TODO - Find accurate way to find vocab_size && num_merges
# vocab_size = 10000 # the desired final vocabulary size
# num_merges = math.isqrt(vocab_size - 348) # substract the number of tokens that we already have

num_merges = 20

ids = list(tokens)

merges = {} # (int, int) -> int * a kind of tree data structure (actually 'forest')
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get) # ignore the error
    idx = 3488 + i
    print(f"merging {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

# print(sorted(((v,k) for k, v in stats.items()), reverse=True))

merging (32, 208) into a new token 3488
merging (208, 190) into a new token 3489
merging (208, 176) into a new token 3490
merging (208, 181) into a new token 3491
merging (209, 130) into a new token 3492
merging (208, 184) into a new token 3493
merging (208, 189) into a new token 3494
merging (209, 129) into a new token 3495
merging (209, 128) into a new token 3496
merging (3489, 208) into a new token 3497
merging (3490, 208) into a new token 3498
merging (209, 131) into a new token 3499
merging (44, 3488) into a new token 3500
merging (3491, 208) into a new token 3501
merging (209, 143) into a new token 3502
merging (209, 140) into a new token 3503
merging (209, 135) into a new token 3504
merging (3488, 178) into a new token 3505
merging (208, 186) into a new token 3506
merging (209, 139) into a new token 3507


In [9]:
print(f"token length: {len(tokens)}")
print(f"ids length: {len(ids)}")
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

token length: 15344
ids length: 9061
compression ratio: 1.69X


Note. the Tokenizer is a completely separate, independent module from the LLM. It has its own training dataset of text (which could be different from that of the LLM), on which you train the vocabulary using the Byte Pair Encoding (BPE).

It then translates back and forth between raw text and sequences of tokens. The LLM later only ever sees the tokens and never directly deals with any text. 


#### Just remember this: More tokens, shorter embedded vector.

### Decoding

Gicven a seqeunce of integers in the range [0, vocab_size], what is the text? 

In [10]:
vocab = { idx: bytes([idx]) for idx in range(256)} # 256 : the size of utf-8 (2^8)
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors='replace')
    return text

print(decode([135]))

�


### Encoding

the other way around: String -> what token?

In [ ]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens) # How many times a token is repeated
        # a bit fancy way
        pair = min(stats, key=lambda p: merges.get(p, float("inf"))) 
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode("hello dudes!"))

print(encode("h")) # It gives us error when just `While True`

[104, 101, 108, 108, 111, 32, 100, 117, 100, 101, 115, 33]
[104]


In [16]:
print(decode(encode("hello")))

hello


### Forced splits using regex patterns (GPT series)

In [ ]:
import regex as re
# \s+(?!\S) - considering space a single chunk
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello! world2, how are you? HOW'S your children doing? I heard you said 'say' yesterday."))

# Never gonna merge some useless cases like space + letter, letter + symbol

['Hello', '!', ' world', '2', ',', ' how', ' are', ' you', '?', ' HOW', "'", 'S', ' your', ' children', ' doing', '?', ' I', ' heard', ' you', ' said', " '", 'say', "'", ' yesterday', '.']


In [24]:
example = """
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens) # How many times a token is repeated
        # a bit fancy way
        pair = min(stats, key=lambda p: merges.get(p, float("inf"))) 
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens
"""

print(re.findall(gpt2pat, example))

['\n', 'def', ' encode', '(', 'text', '):', '\n   ', ' tokens', ' =', ' list', '(', 'text', '.', 'encode', '("', 'utf', '-', '8', '"))', '\n   ', ' while', ' len', '(', 'tokens', ')', ' >=', ' 2', ':', '\n       ', ' stats', ' =', ' get', '_', 'stats', '(', 'tokens', ')', ' #', ' How', ' many', ' times', ' a', ' token', ' is', ' repeated', '\n       ', ' #', ' a', ' bit', ' fancy', ' way', '\n       ', ' pair', ' =', ' min', '(', 'stats', ',', ' key', '=', 'lambda', ' p', ':', ' merges', '.', 'get', '(', 'p', ',', ' float', '("', 'inf', '")))', ' \n       ', ' if', ' pair', ' not', ' in', ' merges', ':', '\n           ', ' break', ' #', ' nothing', ' else', ' can', ' be', ' merged', '\n       ', ' idx', ' =', ' merges', '[', 'pair', ']', '\n       ', ' tokens', ' =', ' merge', '(', 'tokens', ',', ' pair', ',', ' idx', ')', '\n   ', ' return', ' tokens', '\n']


In [28]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print(enc.encode("     hello world!"))

enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("     hello world!"))

[220, 220, 220, 220, 23748, 995, 0]
[257, 24748, 1917, 0]
